# 04 – Sinh tên chủ đề cụm (TF-IDF, spaCy, Gemini)

Phụ trách: Xuân Hoàng (TV5). Notebook chạy từ thư mục gốc repo hoặc từ `notebooks/`.

Cấu hình tốt nhất đã chốt ở phần 4: **BoW + LSA (10 chiều) + K-Means, k = 6** (`config.LSA_COMPONENTS = 10`).

Thứ tự: (1) nạp dữ liệu & bảng kết quả → (2) top 15 từ khóa TF-IDF → (3) 5 bài gần tâm cụm nhất → (4) từ ghép spaCy bổ trợ → (5) sinh tên & mô tả bằng Gemini cho cấu hình tốt nhất (k = 6) → (6) đối chiếu nhãn thật (`topic_vs_label.csv`).

In [1]:
import json
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
from src import config
from src.clustering.kmeans import run_kmeans
from src.evaluation.metrics import contingency
from src.features.lsa import load_features
from src.representation import bow_tfidf
from src.topic_naming.gemini_namer import run_topic_naming
from src.topic_naming.keywords import nearest_articles, top_keywords
from src.topic_naming.spacy_np import top_compound_nouns

pd.set_option("display.max_colwidth", 120)

## 1. Nạp dữ liệu, bảng thí nghiệm và nhãn cụm

In [2]:
exp_df = pd.read_csv(config.TABLES_DIR / "experiments.csv")
labels_df = pd.read_csv(config.TABLES_DIR / "cluster_labels.csv")
df, docs = bow_tfidf.load_corpus()
X_tfidf, v_tfidf = bow_tfidf.load_matrix("tfidf")

REP_DIMS = {"bow_lsa": config.LSA_COMPONENTS, "tfidf_lsa": config.LSA_COMPONENTS, "embedding": config.W2V_SIZE}
exp_k6 = exp_df[
    (exp_df["k"] == 6) & (exp_df["n_components"] == exp_df["representation"].map(REP_DIMS))
].reset_index(drop=True)

# Cấu hình tốt nhất: BoW + LSA (10 chiều) + K-Means, k = 6
X_best = load_features("bow_lsa")
labels_best, centers_best, _ = run_kmeans(X_best, 6)
sub_csv = labels_df[
    (labels_df["representation"] == "bow_lsa")
    & (labels_df["n_components"] == config.LSA_COMPONENTS)
    & (labels_df["model"] == "kmeans")
    & (labels_df["k"] == 6)
]
csv_aligned = df[["id"]].merge(sub_csv[["id", "cluster"]], on="id", how="left")["cluster"].to_numpy()
assert (labels_best == csv_aligned).all(), "Nhãn cụm chạy lại không khớp cluster_labels.csv!"
print("Khớp 100% nhãn cụm với cluster_labels.csv cho cấu hình bow_lsa + kmeans, k = 6.")
exp_k6

Khớp 100% nhãn cụm với cluster_labels.csv cho cấu hình bow_lsa + kmeans, k = 6.


,representation,n_components,model,k,silhouette,davies_bouldin,calinski_harabasz,ari,nmi,purity
0,bow_lsa,10,kmeans,6,0.4961,1.2978,32.3371,0.5155,0.5943,0.7479
1,bow_lsa,10,kmedoids,6,0.4780,1.3125,30.8168,0.5082,0.5854,0.7059
2,tfidf_lsa,10,kmeans,6,0.4871,1.1994,30.4549,0.4655,0.5972,0.6807
3,tfidf_lsa,10,kmedoids,6,0.4708,1.1661,27.7668,0.4641,0.5722,0.7059
4,embedding,80,kmeans,6,0.2886,1.8865,15.5490,0.4723,0.6048,0.6807
5,embedding,80,kmedoids,6,0.2822,1.8317,15.1185,0.3907,0.5463,0.6387


## 2. Trích xuất top 15 từ khóa TF-IDF mỗi cụm

In [3]:
keywords = top_keywords(X_tfidf, v_tfidf, labels_best, n=15)
pd.DataFrame({c: ", ".join(w) for c, w in keywords.items()}, index=["Top 15 từ khóa TF-IDF"]).T

,Top 15 từ khóa TF-IDF
0,"công_ty, nvidia, huang, microsoft, mỹ, hãng, dùng, ceo, sự_cố, nhà, jensen_huang, jensen, usd, apple, công_nghệ"
1,"khán_giả, diễn_viên, phim, nghệ_sĩ, nam, ca_sĩ, nữ, hát, vai, sân_khấu, mang, lời, nhân_vật, con, tên"
2,"bác_sĩ, bệnh_viện, điều_trị, người_bệnh, cơ_thể, bệnh_nhân, bệnh, tăng, nguy_cơ, y_tế, khám, thuốc, tổn_thương, máu,..."
3,"trận, hlv, việt_nam, đội_tuyển, đội, trận_đấu, bóng, giải, asian_cup, đội bóng, thi_đấu, giao_hữu, bóng_đá, clb, cầu..."
4,"đồng, thuế, giá, tiền, nộp, tỉ đồng, tăng, tỉ, ngân_sách, thị_trường, giảm, thanh_tra, nhà_nước, tiêu_dùng, mức"
5,"tỉnh, bộ, quản_lý, tổ_chức, địa_phương, sở, công_an, yêu_cầu, trách_nhiệm, công_nghệ, cơ_quan, ubnd, pháp_luật, số, ..."


## 3. Tìm 5 bài báo gần tâm cụm nhất

In [4]:
near_arts = nearest_articles(X_best, labels_best, centers_best, df, n=5, snippet_words=120)
pd.DataFrame({c: [a["title"] for a in near_arts[c]] for c in sorted(near_arts)}).T.rename(
    columns=lambda i: f"Bài {i + 1}"
)

,Bài 1,Bài 2,Bài 3,Bài 4,Bài 5
0,Nguyên nhân khiến iPhone 18 trễ hẹn tháng 9,"Sự cố Microsoft tê liệt diện rộng: Giới tài chính phải dùng lại điện thoại bàn, máy fax","Amazon ra mắt camera an ninh biết bay, tự động tuần tra trong nhà",Áo khoác da biểu tượng của Jensen Huang thành đề tài bàn luận sau bê bối tình ái,Hé lộ 'thế lực bí ẩn' gây ám ảnh ngày tận thế AI
1,Ju Ji Hoon bình thản với mọi chỉ trích trong 'Hoàng hậu tái hôn',"Làm mới 'Như khúc tình ca', chàng trai được Thái Châu, Quốc Đại cho điểm tuyệt đối",Sắc vóc diễn viên Gen Z Thu Hà Ceri,Phim AI chuyển thể 'Liêu trai chí dị' được khen,Cát Phượng bức xúc trước tin đồn bị bắt vì sử dụng chất cấm
2,Bác sĩ gắp hàng trăm nọc độc ong trên cơ thể người phụ nữ,"Người phụ nữ đặt vòng nâng âm đạo rồi không lấy được, biến chứng nặng nề",Phát hiện polyp đại trực tràng: Khi nào cần lo nguy cơ ung thư?,"Người bệnh tiểu đường có phải kiêng cơm, trái cây? Bác sĩ giải đáp",Nỗ lực giữ thai cho sản phụ từng 5 lần mất con
3,Nhật Bản hủy trận giao hữu với tuyển Việt Nam,Hủy trận giao hữu Nhật Bản - Việt Nam,"Sau trận Nhật Bản, đội tuyển Việt Nam hủy luôn trận giao hữu với đội tuyển Qatar",Chủ tịch FIFA nói gì về World Cup lần đầu có Việt Nam tham dự?,"'Địa chấn' môn cờ tướng, kỳ thủ Việt Nam 18 tuổi giành HCV cá nhân ở giải thế giới"
4,Bỏ quỹ bình ổn xăng dầu: Công cụ nào kìm giá khi có biến động mạnh?,Ngân hàng Nhà nước thanh tra việc giảm lãi suất của các ngân hàng trong tháng 11,"Săn vé miễn phí xem SOOBIN, AMEE, B Ray vào ngày mai tại 'Vibrant HCMC 2026'",Công ty PNJ bị phạt gần 2 tỷ đồng vì khai sai thuế,Cá kèo tự nhiên giá tới nửa triệu đồng/kg vẫn khan hàng
5,Bộ Chính trị: Luân chuyển cán bộ giữa ngành thanh tra với các cơ quan,Thu hồi toàn quốc một lô nước muối sinh lý SATBB do không đạt chất lượng,"Quân khu 7 kiểm tra việc tìm kiếm hài cốt liệt sĩ tại phường Bình Cơ, TP.HCM",TP.HCM thành lập Liên minh Truyền thông đa nền tảng: Lấy lợi ích cộng đồng làm mục tiêu,Quốc hội mở cơ chế thử nghiệm có kiểm soát trong lĩnh vực chứng khoán


## 4. Phân tích từ ghép bổ trợ bằng spaCy (`spacy.blank('vi')`)

In [5]:
spacy_compounds = top_compound_nouns(docs, labels_best, n=10)
pd.DataFrame({
    "TF-IDF (top 10)": {c: ", ".join(keywords[c][:10]) for c in sorted(keywords)},
    "spaCy từ ghép (top 10)": {c: ", ".join(spacy_compounds[c]) for c in sorted(keywords)},
})

,TF-IDF (top 10),spaCy từ ghép (top 10)
0,"công_ty, nvidia, huang, microsoft, mỹ, hãng, dùng, ceo, sự_cố, nhà","công_ty, sự_cố, hoạt_động, dịch_vụ, nhân_viên, công_nghệ, ảnh_hưởng, dự_đoán, đầu_tư, jensen_huang"
1,"khán_giả, diễn_viên, phim, nghệ_sĩ, nam, ca_sĩ, nữ, hát, vai, sân_khấu","diễn_viên, nghệ_sĩ, khán_giả, ca_sĩ, gia_đình, phương_thanh, sân_khấu, tổ_chức, trường_sa, kim_seung_woo"
2,"bác_sĩ, bệnh_viện, điều_trị, người_bệnh, cơ_thể, bệnh_nhân, bệnh, tăng, nguy_cơ, y_tế","bệnh_viện, bác_sĩ, điều_trị, bệnh_nhân, y_tế, người_bệnh, huyết_áp, nguy_cơ, khu_vực, cơ_thể"
3,"trận, hlv, việt_nam, đội_tuyển, đội, trận_đấu, bóng, giải, asian_cup, đội bóng","việt_nam, đội_tuyển, trận_đấu, asian_cup, nhật_bản, bóng_đá, giao_hữu, thi_đấu, hợp_đồng, man_utd"
4,"đồng, thuế, giá, tiền, nộp, tỉ đồng, tăng, tỉ, ngân_sách, thị_trường","ngân_sách, nhà_nước, ngân_hàng, doanh_nghiệp, lãi_suất, cơ_quan, quy_định, thị_trường, tài_chính, quốc_hội"
5,"tỉnh, bộ, quản_lý, tổ_chức, địa_phương, sở, công_an, yêu_cầu, trách_nhiệm, công_nghệ","công_nghệ, tổ_chức, việt_nam, dữ_liệu, quản_lý, bộ_trưởng, cơ_sở, sáng_tạo, pháp_luật, hoạt_động"


## 5. Sinh tên chủ đề và mô tả bằng Gemini (BoW + LSA + K-Means, k = 6)

In [6]:
json_path = config.TOPIC_NAMES_DIR / "bow_lsa_kmeans_k6.json"
if json_path.exists():
    best_names = json.loads(json_path.read_text(encoding="utf-8"))
else:
    best_names, _ = run_topic_naming(k=6)
    config.TOPIC_NAMES_DIR.mkdir(parents=True, exist_ok=True)
    json_path.write_text(json.dumps(best_names, ensure_ascii=False, indent=2), encoding="utf-8")

pd.DataFrame.from_dict(best_names, orient="index")[["name", "description"]]

,name,description
0,Công nghệ toàn cầu,"Tổng hợp các chuyển động nổi bật của các tập đoàn công nghệ lớn như Apple, Microsoft, Amazon và Nvidia. Các bài viết..."
1,Giải trí nghệ thuật,"Tổng hợp các tin tức nổi bật xoay quanh đời sống nghệ sĩ, phim ảnh và các chương trình nghệ thuật sân khấu trong nướ..."
2,Y tế sức khỏe,"Tổng hợp các ca điều trị bệnh lý, cấp cứu ngoại khoa cùng lời khuyên y khoa từ bác sĩ chuyên khoa. Nội dung giúp ngư..."
3,Bóng đá Việt Nam,"Tổng hợp tin tức về các trận đấu giao hữu, kế hoạch chuẩn bị và các giải đấu quốc tế liên quan đến đội tuyển bóng đá..."
4,Kinh tế thị trường,"Cập nhật các biến động về giá cả hàng hóa, chính sách tài chính ngân hàng, thuế và hoạt động khuyến mãi tiêu dùng."
5,Quản lý nhà nước,"Tổng hợp các chỉ đạo, chính sách mới và hoạt động quản lý, điều hành của các cơ quan chức năng trên nhiều lĩnh vực đ..."


## 6. Đối chiếu tên chủ đề với nhãn thật (`topic_vs_label.csv`)

In [7]:
ct = contingency(df["true_label"], labels_best)
display(ct)

rows = []
for cid in ct.index:
    c = int(cid)
    row = ct.loc[cid]
    so_bai = int(row.sum())
    chu_de_chiem_da_so = str(row.idxmax())
    rows.append({
        "cluster": c,
        "gemini_name": best_names[str(c)]["name"],
        "chu_de_chiem_da_so": chu_de_chiem_da_so,
        "ti_le": round(float(row.max() / so_bai), 4),
    })
vs_df = pd.DataFrame(rows)
config.TABLES_DIR.mkdir(parents=True, exist_ok=True)
vs_df.to_csv(config.TABLES_DIR / "topic_vs_label.csv", index=False, encoding="utf-8")
vs_df

Chủ đề thật,Công nghệ,Giải trí,Kinh tế,Sức khỏe,Thể thao,Thời sự
Cụm,,,,,,
0,12,1,6,0,1,0
1,1,18,2,0,0,0
2,0,0,0,18,1,3
3,0,0,1,0,17,1
4,1,0,9,0,0,1
5,6,1,2,2,0,15


,cluster,gemini_name,chu_de_chiem_da_so,ti_le
0,0,Công nghệ toàn cầu,Công nghệ,0.6000
1,1,Giải trí nghệ thuật,Giải trí,0.8571
2,2,Y tế sức khỏe,Sức khỏe,0.8182
3,3,Bóng đá Việt Nam,Thể thao,0.8947
4,4,Kinh tế thị trường,Kinh tế,0.8182
5,5,Quản lý nhà nước,Thời sự,0.5769


## Nhận xét

- Cấu hình tốt nhất **`bow_lsa` (10 chiều) + `kmeans`, `k = 6`** tạo ra 6 cụm tách biệt rõ, mỗi cụm đều gắn được tên chủ đề tương ứng trực tiếp với một nhóm tin tức nổi bật.
- **Vai trò bổ trợ của `spacy.blank("vi")`:** Đếm từ ghép có `_` giúp đối chiếu trực tiếp với từ khóa TF-IDF và làm rõ các cụm danh từ/thuật ngữ tiếng Việt đặc trưng cho từng cụm.
- **Sinh tên chủ đề bằng Gemini:** `name_cluster` đọc `GEMINI_API_KEY` từ `.env` và gọi Gemini (`gemini-3.5-flash`, không đưa danh sách 6 nhãn thật vào prompt) để sinh JSON `{name, description}`, sau đó lưu sẵn vào `outputs/topic_names/bow_lsa_kmeans_k6.json` và `outputs/tables/topic_vs_label.csv` cho app Streamlit đọc offline.